# ComfyUI on Google Colab (Free GPU)

Run [ComfyUI](https://github.com/comfyanonymous/ComfyUI) — the open-source Stable Diffusion / SDXL frontend — on Colab's **free** GPU, then reach it from your local browser through a free Cloudflare tunnel.

> ⚠️ **Colab free tier is time-limited** (a few hours per session) and disconnects when idle. For long jobs use the backup free APIs (Pollinations / Stable Horde) or another free GPU notebook (Kaggle, Lightning AI).

At the end, copy the `https://<random>.trycloudflare.com` URL into the platform's `COMFYUI_URL`.

In [ ]:
# 1) Check that we actually got a GPU
!nvidia-smi || echo "NO GPU DETECTED - set Runtime > Change runtime type > Hardware accelerator > GPU (T4), then re-run this cell."

In [ ]:
# 2) Clone ComfyUI and install its dependencies
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt

# Optional but handy: ComfyUI-Manager for installing nodes/models from the UI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager
%cd /content/ComfyUI

In [ ]:
# 3) Download a free base model — SDXL base 1.0 (CreativeML OpenRAIL++-M license)
!mkdir -p /content/ComfyUI/models/checkpoints
%cd /content/ComfyUI/models/checkpoints
!wget -q --show-progress -O sd_xl_base_1.0.safetensors https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors

In [ ]:
# 4) Install cloudflared for the free tunnel — no Cloudflare account needed
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 5) Launch ComfyUI in the background and wait until it is ready
import subprocess, time, urllib.request

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI",
)

for i in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as r:
            print("✅ ComfyUI is up:", r.read().decode()[:300])
            break
    except Exception:
        print(f"waiting for ComfyUI... {i + 1}")
        time.sleep(3)
else:
    print("❌ ComfyUI did not start — check the install cell output above.")

In [ ]:
# 6) Start the free Cloudflare tunnel and print the public URL
!pkill -f cloudflared || true
!nohup cloudflared tunnel --url http://127.0.0.1:8188 > /content/cloudflared.log 2>&1 &

import time

url = None
for _ in range(30):
    time.sleep(2)
    try:
        log = open("/content/cloudflared.log").read()
    except FileNotFoundError:
        continue
    if "trycloudflare.com" in log:
        for line in log.splitlines():
            if "trycloudflare.com" in line:
                url = line.strip()
        break

print("🔗 YOUR COMFYUI URL:", url or "still starting — run:  !cat /content/cloudflared.log")

## Troubleshooting

- **No GPU / everything slow:** Runtime > Change runtime type > Hardware accelerator > GPU (T4).
- **Colab disconnected:** re-run the install → launch → tunnel cells; the URL changes on every run, so paste the new one into `COMFYUI_URL`.
- **Out of memory:** lower the resolution (e.g. 768x768) or switch to a smaller model (SD 1.5) instead of SDXL.
- **Tunnel URL rotates:** `trycloudflare.com` quick tunnels are temporary by design. For a stable URL use a named Cloudflare tunnel with your own domain.
- **Keep the tab alive:** the tunnel dies when the Colab session ends.